# EARC Pipeline — Full Pipeline Test (Layers 1-13)

This notebook runs the **complete** EARC pipeline on Colab using the corpus artifacts stored in Google Drive:

**Retrieval (1-3) → Scoring (4-6) → Selection (7-10) → Generation (11-13)**

**What you get per query:**
- Module 1: query analysis + retrieved/segmented sentences
- Module 2: query-first embeddings + multi-signal scores + redundancy removal
- Module 3: reasoning-graph bridges + token-budget selection + diversity guard + sufficiency
- Module 4: cited prompt construction + answer generation + grounding verification

> Run the cells top-to-bottom. Steps 1-5 are one-time setup; Steps 6+ are the actual tests.
>
> **Generation backend:** by default Module 4 uses the dependency-free **extractive** backend (no model download, works offline). Step 5 shows how to switch to `flan-t5`, OpenAI, or Ollama.


## Step 1 — Mount Google Drive
Your FAISS index, BM25 index, chunk shards and metadata shards live here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Get the code

Pick **one** option below.

- **Option A (GitHub):** clone the repo. Make sure you have **pushed** the latest Module 2/3 changes first, otherwise you'll get the old stubs.
- **Option B (Drive):** if you keep the project folder in Drive, set `REPO_DIR` to that path and skip the clone.

In [ ]:
import os, sys

# ---- Option A: clone from GitHub (uncomment) ----
# !rm -rf /content/earc-pipeline
# !git clone https://github.com/anaghadk/EARC_Pipeline.git /content/earc-pipeline
# REPO_DIR = '/content/earc-pipeline'

# ---- Option B: use a copy already in Drive (edit path) ----
REPO_DIR = '/content/drive/MyDrive/RAG_Project/earc-pipeline'

assert os.path.isdir(REPO_DIR), f'REPO_DIR not found: {REPO_DIR}'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Using repo at:', REPO_DIR)
print(os.listdir(REPO_DIR))

## Step 3 — Install dependencies
Installs the pinned requirements and the spaCy English model. Safe to re-run.

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print('Dependencies installed.')

## Step 4 — Point to your corpus artifacts

These are the same files Module 1 already uses. Edit the paths if your Drive layout differs.
Expected layout:
```
RAG_Project/
  faiss.index
  bm25.pkl
  chunks/      (chunks_*.pkl)
  metadata/    (metadata_*.pkl)
```

In [ ]:
from pathlib import Path

DRIVE_BASE   = Path('/content/drive/MyDrive/RAG_Project')
FAISS_PATH   = DRIVE_BASE / 'faiss.index'
BM25_PATH    = DRIVE_BASE / 'bm25.pkl'
CHUNKS_DIR   = DRIVE_BASE / 'chunks'
METADATA_DIR = DRIVE_BASE / 'metadata'

for p in [FAISS_PATH, BM25_PATH, CHUNKS_DIR, METADATA_DIR]:
    print(('OK   ' if p.exists() else 'MISSING ') + str(p))

## Step 5 — Initialise the full pipeline (Modules 1-4)

`EARCPipeline` loads all artifacts once and chains:
Retrieval (1-3) → Scoring (4-6) → Selection (7-10) → Generation (11-13).

First run takes a minute (downloads embedding model + loads FAISS/BM25).

**Choosing the generation backend** (optional). The default `extractive` backend needs no
extra downloads and always works. To use a real LLM instead, edit `config.py`'s
`CONFIG["generation"]["backend"]` to one of:
- `"transformers"` — local `flan-t5` (auto-downloads the model)
- `"openai"` — needs `OPENAI_API_KEY` in the environment
- `"ollama"` — needs a running Ollama server

The **version guard** at the end fails loudly if Colab is still running OLD code
(before Modules 2/3/4 were wired in). If it trips: re-sync the repo and restart the runtime.


In [ ]:
from pipeline import EARCPipeline

pipe = EARCPipeline(
    faiss_path=FAISS_PATH,
    bm25_path=BM25_PATH,
    chunks_dir=CHUNKS_DIR,
    metadata_dir=METADATA_DIR,
)

# Version guard: confirm Colab is running the UPDATED code (Modules 2, 3 & 4).
# If any assert fails, Colab loaded an OLD copy of the repo.
# Fix: re-sync the code (git push + re-clone, or replace the Drive copy),
# then Runtime > Restart runtime and re-run from Step 5.
assert hasattr(pipe, 'scoring_pipeline'), (
    'OLD CODE: EARCPipeline has no scoring_pipeline. Re-sync repo + restart runtime.'
)
assert hasattr(pipe, 'selection_pipeline'), (
    'OLD CODE: EARCPipeline has no selection_pipeline. Re-sync repo + restart runtime.'
)
assert hasattr(pipe, 'generation_pipeline'), (
    'OLD CODE: EARCPipeline has no generation_pipeline. Re-sync repo + restart runtime.'
)
print('\nEARCPipeline ready — Modules 1-4 wired in (Layers 1-13). Code version OK.')


## Step 6 — Pretty-printer for all 4 modules
A small helper that shows each module's contribution for a single query, ending with the generated, cited answer and its grounding check.


In [ ]:
def show_result(result, top_k_scored=8, top_k_selected=12):
    qi = result['query_info']
    print('=' * 80)
    print('QUERY:', result['query'])
    print('=' * 80)

    # ---- Module 1: Query analysis + retrieval ----
    print('\n[MODULE 1] Query Analysis & Retrieval')
    print('  query_type   :', qi['query_type'])
    print('  has_negation :', qi['has_negation'])
    print('  keywords     :', qi['keywords'])
    print('  entities     :', qi['entities'])
    print('  scored sents :', len(result['sentences']))

    # ---- Module 2: Scoring ----
    print('\n[MODULE 2] Top scored sentences (after embedding + dedup)')
    top = sorted(result['sentences'], key=lambda s: s.final_score, reverse=True)[:top_k_scored]
    for i, s in enumerate(top, 1):
        print(f'  {i:2d}. score={s.final_score:.4f}  '
              f'(sim={s.semantic_score:.2f} ev={s.evidence_score:.2f} '
              f'evd={s.evidentiality_score:.2f} dens={s.claim_density_score:.2f})  '
              f'[{s.dataset}/{s.doc_id}]')
        print('      ', s.text[:160])

    # ---- Module 3: Selection ----
    if 'selected_sentences' not in result:
        print('\n[MODULE 3] MISSING. You are running OLD code (no Module 2/3 output).')
        print('  Fix: re-sync the repo to Colab, then Runtime > Restart runtime.')
        return
    sel = result['selected_sentences']
    print(f'\n[MODULE 3] Selected evidence: {len(sel)} sentences')
    for i, s in enumerate(sel[:top_k_selected], 1):
        print(f'  {i:2d}. score={float(s["score"]):.4f}  bridge={s["is_bridge"]}  '
              f'[{s.get("dataset", "?")}/{s["doc_id"]}]')
        print('      ', s['text'][:160])

    # ---- Selection stats ----
    stats = result['selection_stats']
    print('\n[MODULE 3] Stats')
    print('  reasoning :', stats.get('reasoning'))
    print('  budget    :', stats.get('budget'))
    print('  diversity :', {k: stats.get('diversity', {}).get(k) for k in
          ('coverage_ratio', 'keyword_coverage_ratio', 'sentence_swaps', 'coverage_improved')})
    print('  sufficiency:', {k: stats.get('sufficiency', {}).get(k) for k in
          ('is_sufficient', 'required_evidence', 'final_evidence_count',
           'expansions', 'stopped_reason')})

    # ---- Module 4: Generation ----
    if 'answer' not in result:
        print('\n[MODULE 4] MISSING. You are running OLD code (no Module 4 output).')
        print('  Fix: re-sync the repo to Colab, then Runtime > Restart runtime.')
        return
    gen = result['generation']
    ver = gen['verification']
    print('\n[MODULE 4] Generated answer')
    print('  backend  :', gen['backend'])
    print('  answer   :', result['answer'])
    print('  grounded :', ver['grounded'],
          f"(faithfulness={ver['faithfulness']}, mean_overlap={ver['mean_overlap']})")
    print('  citations:', ver['distinct_citations'],
          '| invalid:', ver['invalid_citations'])
    if ver['unsupported_sentences']:
        print('  UNSUPPORTED:', ver['unsupported_sentences'])
    print('\n  sources cited:')
    for c in gen['citations']:
        print(f"    [{c['marker']}] ({c['dataset']}/{c['doc_id']}) {c['text'][:120]}")
    print('=' * 80)


## Step 7 — Run a single query
Change the text and re-run to test interactively.

In [ ]:
result = pipe.run('Who invented the telephone?')
show_result(result)

## Step 8 — Batch test across query types
Covers factoid, multi-hop and negation queries so you can confirm classification + selection behave per type.

In [ ]:
test_queries = [
    'Who invented the telephone?',
    'What did Marie Curie and Albert Einstein both contribute to physics?',
    'What countries are not members of NATO?',
    'How does photosynthesis work?',
]

for q in test_queries:
    res = pipe.run(q)
    show_result(res, top_k_scored=4, top_k_selected=6)
    print('\n')

## Step 9 — (Optional) Inspect each module in isolation
Useful for debugging: runs all four modules step-by-step and shows the intermediate hand-off so you can confirm each boundary is healthy.


In [ ]:
query = 'What did Marie Curie and Albert Einstein both contribute to physics?'

# --- Module 1 ---
sentences, query_info = pipe.retrieval_layer.retrieve(query)
print('M1 ->', len(sentences), 'sentences | type =', query_info['query_type'])
assert sentences and sentences[0].embedding is None, 'M1 should leave embedding=None'

# --- Module 2 ---
scored = pipe.scoring_pipeline.run(query_info, sentences)
print('M2 ->', len(scored), 'scored+deduped | embedding set =', scored[0].embedding is not None)
records = pipe.scoring_pipeline.to_selection_records(scored)
print('M2 -> adapter records:', len(records), '| keys =', sorted(records[0].keys())[:6], '...')

# --- Module 3 ---
selection = pipe.selection_pipeline.run(query_info, records)
print('M3 -> selected =', len(selection['selected_sentences']),
      '| candidates =', len(selection['candidate_sentences']))
print('M3 -> stats keys:', list(selection['stats'].keys()))

# --- Module 4 ---
generation = pipe.generation_pipeline.generate(query_info, selection['selected_sentences'])
print('M4 -> backend =', generation['backend'],
      '| grounded =', generation['verification']['grounded'])
print('M4 -> answer  :', generation['answer'])


## Step 10 — (Optional) Export full pipeline output
Saves the complete Module 1-4 output (including the generated answer + verification) to Drive as a pickle, e.g. for evaluation or sharing.


In [ ]:
import pickle, re
from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/RAG_Project/earc_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export_full(query):
    result = pipe.run(query)
    slug = re.sub(r'[^a-z0-9]+', '_', query.lower())[:40]
    out_path = OUTPUT_DIR / f'earc_{slug}.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump({
            'query': result['query'],
            'query_info': result['query_info'],
            'selected_sentences': result['selected_sentences'],
            'candidate_sentences': result['candidate_sentences'],
            'selection_stats': result['selection_stats'],
            'answer': result['answer'],
            'generation': result['generation'],
        }, f)
    print(f'answer: {result["answer"]}')
    print(f'Saved {len(result["selected_sentences"])} selected sentences + answer -> {out_path.name}')
    return out_path

export_full('Who invented the telephone?')


## Step 11 — (Optional) Try a real LLM backend for Module 4

The default backend is `extractive` (deterministic, no download). To compare against a
real generative model **without restarting the runtime**, swap the generator below.

- `transformers` downloads `flan-t5-base` (a few hundred MB) the first time.
- `openai` needs `OPENAI_API_KEY` set; `ollama` needs a running Ollama server.

If a backend can't load (missing dep / key / server), Module 4 automatically falls back
to `extractive`, so this cell never hard-crashes.


In [ ]:
from generation.answer_generator import AnswerGenerator

# Swap the generator on the existing pipeline. Try: 'transformers', 'openai', 'ollama'.
pipe.generation_pipeline.generator = AnswerGenerator(backend='transformers')

res = pipe.run('Who invented the telephone?')
print('backend  :', res['generation']['backend'])
print('answer   :', res['answer'])
print('grounded :', res['generation']['verification']['grounded'],
      '| faithfulness =', res['generation']['verification']['faithfulness'])

# Restore the default extractive backend afterwards (optional):
# pipe.generation_pipeline.generator = AnswerGenerator(backend='extractive')


## Step 12 — (Optional) Quantitative evaluation

Runs the full pipeline over a sample of labelled QA pairs and reports:
- **Exact Match** and **token-level F1** vs. gold answers
- **Contains-gold** (lenient: gold span found inside the generated answer)
- **Grounded rate** and **faithfulness** (from Layer 13)
- **Compression ratio** — selected tokens ÷ retrieved tokens (the EARC efficiency metric; lower = more compression)
- mean evidence count and latency, broken down per dataset

Point `QA_DIR` at your `qa_pairs` shard folder in Drive. Start with a small
`SAMPLE_SIZE` to confirm it works, then scale up.


In [ ]:
import logging
from pathlib import Path
from evaluation import run_evaluation
from generation.answer_generator import AnswerGenerator

# Silence the harmless "Token indices sequence length is longer..." tokenizer
# warning emitted by the budget layer when counting tokens of long chunks.
logging.getLogger('transformers.tokenization_utils_base').setLevel(logging.ERROR)

# Generation backend for evaluation.
# 'extractive' (default) is FAST and dependency-free, but returns full
# sentences so it scores low on Exact Match / F1 (which expect short gold
# spans). For a FAIR EM/F1, set EVAL_BACKEND='transformers' (flan-t5) -- but
# it is much slower on CPU, so keep SAMPLE_SIZE small or use a GPU runtime.
EVAL_BACKEND = 'extractive'      # 'extractive' | 'transformers' | 'openai' | 'ollama'
pipe.generation_pipeline.generator = AnswerGenerator(backend=EVAL_BACKEND)

# Folder containing the qa_*.pkl shards produced by data/rag_pipeline.py
QA_DIR = Path('/content/drive/MyDrive/RAG_Project/qa_pairs')

SAMPLE_SIZE = 25                 # start small; raise to 500+ for a real run
# Skip the 'nq' dataset by default: its answer rows have doc_id=None (no
# corpus linkage), so gold answers are usually absent from the corpus and
# EM is structurally ~0. Evaluate against the corpus-linked datasets.
DATASETS = ['hotpot', 'trivia']  # set to None for all, incl. nq
OUTPUT = Path('/content/drive/MyDrive/RAG_Project/earc_outputs/eval_results.pkl')

assert QA_DIR.exists(), f'QA_DIR not found: {QA_DIR}'

def _progress(done, total):
    print(f'  evaluated {done}/{total}', end='\r')

results = run_evaluation(
    pipe,
    qa_dir=QA_DIR,
    sample_size=SAMPLE_SIZE,
    datasets=DATASETS,
    output_path=OUTPUT,
    progress_callback=_progress,
)
print('\nDone. Saved ->', OUTPUT.name)


In [ ]:
def show_eval(results):
    summary = results['summary']
    o = summary['overall']
    print('=' * 70)
    print(f"EVALUATION  ({o['count']} examples, {summary['n_errors']} errors)")
    print(f"  backend: {results['config']['generation_backend']}")
    print('=' * 70)
    print(f"  Exact Match     : {o['exact_match']:.3f}")
    print(f"  F1              : {o['f1']:.3f}")
    print(f"  Contains gold   : {o['contains_gold']:.3f}")
    print(f"  Ans in retrieved: {o['answer_in_retrieved']:.3f}  (retrieval recall)")
    print(f"  Ans in selected : {o['answer_in_selected']:.3f}  (selection recall)")
    print(f"  Grounded rate   : {o['grounded_rate']:.3f}")
    print(f"  Faithfulness    : {o['faithfulness']:.3f}")
    print(f"  Compression     : {o['compression_ratio']:.3f}  "
          f"({o['mean_selected_tokens']:.0f}/{o['mean_retrieved_tokens']:.0f} tokens)")
    print(f"  Mean selected   : {o['mean_selected']:.1f} sentences")
    print(f"  Mean latency    : {o['mean_latency_sec']:.2f}s")

    print('\n  Per dataset:')
    for ds, s in summary['per_dataset'].items():
        print(f"    {ds:<10} n={s['count']:<4} EM={s['exact_match']:.3f} "
              f"F1={s['f1']:.3f} ground={s['grounded_rate']:.3f} "
              f"compress={s['compression_ratio']:.3f}")

    if summary['n_errors']:
        print('\n  First errors:')
        for e in summary['errors'][:5]:
            print(f"    {e.get('question_id')}: {e['error']}")
    print('=' * 70)

show_eval(results)
